# To explore filtering throughput dataset and getting its details (number of samples) <br>
Date: 14/02/2024

## Imports

In [1]:
import pandas as pd
import numpy as np 
import math
import os
import glob
import tensorflow as tf
from tqdm import tqdm
from itertools import product
from scipy import special

def q_func(x):
    q = 0.5 - 0.5*special.erf(x / np.sqrt(2))
    return q

def friis_calc(P,freq,dist,ple):
    '''
    Friis path loss equation
    P = Tx transmit power
    freq = Signal frequency
    dist = Transmission distance
    ple = Path loss exponent
    '''
    propagation_speed = 299792458
    l = propagation_speed / freq
    h_pl = P * l**2 / (16*math.pi**2)
    P_Rx = h_pl * dist**(-ple)
    return P_Rx

def plos_calc(h_dist, height_tx, height_rx, env='suburban'):
    '''
    % This function implements the LoS probability model from the paper
    % "Blockage Modeling for Inter-layer UAVs Communications in Urban
    % Environments" 
    % param h_dist    : horizontal distance between Tx and Rx (m)
    % param height_tx : height of Tx
    % param height_rx : height of Rx
    '''
    if env == 'suburban':
        a1 = 0.1
        a2 = 7.5e-4
        a3 = 8
    
    delta_h = height_tx - height_rx
    # pow_factor = 2 * h_dist * math.sqrt(a1*a2/math.pi) + a1 # NOTE: Use this pow_factor if assuming PPP building dist.
    pow_factor = h_dist * math.sqrt(a1*a2) # NOTE: Use this pow_factor if assuming ITU-R assumptions.
    if delta_h == 0:
        p = (1 - math.exp((-(height_tx)**2) / (2*a3**2))) ** pow_factor
    else:
        delta_h = abs(delta_h)
        p = (1 - (math.sqrt(2*math.pi)*a3 / delta_h) * abs(q_func(height_tx/a3) - q_func(height_rx/a3))) ** pow_factor
    return p

def sinr_lognormal_approx(h_dist, height, env='suburban'):
    '''
    To approximate the SNR from signal considering multipath fading and shadowing
    Assuming no interference due to CSMA, and fixed noise
    Inputs:
    h_dist = Horizontal Distance between Tx and Rx
    height = Height difference between Tx and Rx
    env = The operating environment (currently only suburban supported)
    '''
    # Signal properties
    P_Tx_dBm = 20 # Transmit power of 
    P_Tx = 10**(P_Tx_dBm/10) / 1000
    freq = 2.4e9 # Channel frequency (Hz)
    noise_dBm = -86
    noise = 10**(noise_dBm/10) / 1000
    if env == "suburban":
        # ENV Parameters Constants ----------------------------------
        # n_min = 2
        # n_max = 2.75
        # K_dB_min = 7.8
        # K_dB_max = 17.5
        # K_min = 10**(K_dB_min/10)
        # K_max = 10**(K_dB_max/10)
        # alpha = 11.25 # Env parameters for logarithm std dev of shadowing 
        # beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        n_min = 2
        n_max = 2.75
        K_dB_min = 1.4922
        K_dB_max = 12.2272
        K_min = 10**(K_dB_min/10)
        K_max = 10**(K_dB_max/10)
        alpha = 11.1852 # Env parameters for logarithm std dev of shadowing 
        beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        # -----------------------------------------------------------
    # Calculate fading parameters
    PLoS = plos_calc(h_dist, 0, height, env='suburban')
    theta_Rx = math.atan2(height, h_dist) * 180 / math.pi # Elevation angle in degrees
    ple = (n_min - n_max) * PLoS + n_max # Path loss exponent
    sigma_phi_dB = alpha*math.exp(-beta*theta_Rx)
    sigma_phi = 10**(sigma_phi_dB/10) # Logarithmic std dev of shadowing
    K = K_min * math.exp(math.log(K_max/K_min) * PLoS**2)
    omega = 1 # Omega of NCS (Rician)
    dist = math.sqrt(h_dist**2 + height**2)
    P_Rx = friis_calc(P_Tx, freq, dist, ple)
    # Approximate L-NCS RV (which is the SNR) as lognormal
    eta = math.log(10) / 10
    mu_phi = 10*math.log10(P_Rx)
    E_phi = math.exp(eta*mu_phi + eta**2*sigma_phi**2/2) # Mean of shadowing RV
    var_phi = math.exp(2*eta*mu_phi+eta**2*sigma_phi**2)*(math.exp(eta**2*sigma_phi**2)-1) # Variance of shadowing RV
    E_chi = (special.gamma(1+1)/(1+K))*special.hyp1f1(-1,1,-K)*omega
    var_chi = (special.gamma(1+2)/(1+K)**2)*special.hyp1f1(-2,1,-K)*omega**2 - E_chi**2
    E_SNR = E_phi * E_chi / noise # Theoretical mean of SINR
    var_SNR = ((var_phi+E_phi**2)*(var_chi+E_chi**2) - E_phi**2 * E_chi**2) / noise**2
    std_dev_SNR = math.sqrt(var_SNR)
    # sigma_ln = math.sqrt(math.log(var_SNR/E_SNR**2 + 1))
    # mu_ln = math.log(E_SNR) - sigma_ln**2/2
    return E_SNR, std_dev_SNR

def norm_MCS(mcs_index):
    return 2*mcs_index/7 - 1

def get_measured_throughput(sim_root_path, link="Downlink", single_path = False):
    '''
    Function to load the processed measured throughput data from CSV files stored in different subdirs in sim_root_path
    Modified: The throughput files for each UAV and the GCS are stored separately (rather than single Uplink/Downlink) and separated by runs.
    '''
    assert link in ["Downlink", "Uplink", "Video"], 'link must be one of "Downlink", "Uplink", "Video"'
    df_list = []
    if single_path:
        scenario_list = [sim_root_path]
    else:
        scenario_list = [f.path for f in os.scandir(sim_root_path) if f.is_dir()] # Get list of "unique" scenarios
    for scenario in tqdm(scenario_list):
        # Get the measured throughput samples for UL/DL/Vid under this scenario
        if link == "Downlink":
            throughput_files = glob.glob(os.path.join(scenario, "Run-*_Downlink_Throughput.csv"))
        elif link == "Uplink":
            throughput_files = glob.glob(os.path.join(scenario, "Run-*_Uplink_Throughput.csv"))
        elif link == "Video":
            throughput_files = glob.glob(os.path.join(scenario, "Run-*_Video_Throughput.csv"))
        for file in throughput_files:
            measured_df = pd.read_csv(file)
            df_list.append(measured_df)
    return pd.concat(df_list)

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13.0), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26.0), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39.0), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52.0), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65.0), "MCS"] = 7 # MCS Index 0

    return df

2024-11-25 15:48:42.867604: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-25 15:48:43.128661: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-25 15:48:43.191725: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-25 15:48:43.191745: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if yo

## Find Critical Distance Using DNN For Reliability (v1) </br>
Takes first occurence of h_dist that did not satisfy reliability threshold as crit_distance

In [ ]:
DL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_dl/model.010-0.2039.h5"
UL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_ul/model.010-0.1202.h5"
VID_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_vid/model.010-0.2581.h5"

uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()

'''Load the DNN reliability prediction models for defining regions of reliability under different MCS and USI'''
dl_model = tf.keras.models.load_model(DL_MODEL_PATH, compile=False)
dl_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
ul_model = tf.keras.models.load_model(UL_MODEL_PATH, compile=False)
ul_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
vid_model = tf.keras.models.load_model(VID_MODEL_PATH, compile=False)
vid_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
max_mean_sinr = 10*math.log10(1123) # The max mean SINR calculated at (0,60) is 1122.743643457063 (linear)
max_std_dev_sinr = 10*math.log10(466) # The max std dev SINR calculated at (0,60) is 465.2159856885714 (linear)
min_mean_sinr = 10*math.log10(0.2) # The min mean SINR calculated at (1200,60) is 0.2251212887895188 (linear)
min_std_dev_sinr = 10*math.log10(0.7) # The min std dev SINR calculated at (1200,300) is 0.7160093126585219 (linear)
uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]
horizontal_dist = np.linspace(0, 1200, 121, endpoint=True)
reliability_th = 0.99 # Threshold for reliability value
crit_dist_list = []
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    for height in heights:
        dl_done = 0
        ul_done = 0 
        vid_done = 0
        for h_dist in horizontal_dist:
            m, s = sinr_lognormal_approx(h_dist, height)
            m = 2*(10*math.log10(m)-min_mean_sinr)/(max_mean_sinr-min_mean_sinr) - 1
            s = 2*(10*math.log10(s)-min_std_dev_sinr)/(max_std_dev_sinr-min_std_dev_sinr) - 1
            # Check DL reliability
            if dl_done == 0:
                dl_reliability = dl_model.predict([[m, s, uav_send_int_norm[usi], norm_MCS(mcs)]])[0][0]
                dl_h_dist = h_dist # Store the latest h_dist in reliable region
                if dl_reliability < reliability_th:
                    dl_done = 1
                    crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "DL", "Height": height, "Critical_Distance": dl_h_dist})
            # Check UL reliability
            if ul_done == 0:
                ul_reliability = ul_model.predict([[m, s, uav_send_int_norm[usi], norm_MCS(mcs)]])[0][0]
                ul_h_dist = h_dist # Store the latest h_dist in reliable region
                if ul_reliability < reliability_th:
                    ul_done = 1
                    crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "UL", "Height": height, "Critical_Distance": ul_h_dist})
            # Check VID reliability
            if vid_done == 0:
                vid_reliability = vid_model.predict([[m, s, uav_send_int_norm[usi], norm_MCS(mcs)]])[0][0]
                vid_h_dist = h_dist # Store the latest h_dist in reliable region
                if vid_reliability < reliability_th:
                    vid_done = 1
                    crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "VID", "Height": height, "Critical_Distance": vid_h_dist})
crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("Critical_Distances_DNN_Predictions.csv", index=False)

## Find Critical Distance Using DNN For Reliability Version 2 </br>
Takes last occurence of h_dist that did not satisfy reliability threshold as crit_distance

In [ ]:
DL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_dl/model.010-0.2039.h5"
UL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_ul/model.010-0.1202.h5"
VID_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_vid/model.010-0.2581.h5"

'''Load the DNN reliability prediction models for defining regions of reliability under different MCS and USI'''
dl_model = tf.keras.models.load_model(DL_MODEL_PATH, compile=False)
dl_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
ul_model = tf.keras.models.load_model(UL_MODEL_PATH, compile=False)
ul_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
vid_model = tf.keras.models.load_model(VID_MODEL_PATH, compile=False)
vid_model.compile(optimizer='adam', 
                loss={'packet_state': 'categorical_crossentropy'},
                metrics={'packet_state': 'accuracy'})
max_mean_sinr = 10*math.log10(1123) # The max mean SINR calculated at (0,60) is 1122.743643457063 (linear)
max_std_dev_sinr = 10*math.log10(466) # The max std dev SINR calculated at (0,60) is 465.2159856885714 (linear)
min_mean_sinr = 10*math.log10(0.2) # The min mean SINR calculated at (1200,60) is 0.2251212887895188 (linear)
min_std_dev_sinr = 10*math.log10(0.7) # The min std dev SINR calculated at (1200,300) is 0.7160093126585219 (linear)
uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]
horizontal_dist = np.linspace(0, 1200, 121, endpoint=True)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
reliability_th = 0.99 # Threshold for reliability value
crit_dist_list = []
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    for height in heights:
        mean_sinr = []
        std_dev_sinr = []
        for h_dist in horizontal_dist:
            m, s = sinr_lognormal_approx(h_dist, height)
            m = 2*(10*math.log10(m)-min_mean_sinr)/(max_mean_sinr-min_mean_sinr) - 1
            s = 2*(10*math.log10(s)-min_std_dev_sinr)/(max_std_dev_sinr-min_std_dev_sinr) - 1
            mean_sinr.append(m)
            std_dev_sinr.append(s)

        inputs = np.hstack((np.array(mean_sinr).reshape(-1,1), np.array(std_dev_sinr).reshape(-1,1), 
                            np.ones((len(horizontal_dist),1))*uav_send_int_norm[usi], np.ones((len(horizontal_dist),1))*norm_MCS(mcs)))
        # Evaluate DL reliability at each horizontal_dist
        dl_predictions = dl_model.predict(inputs)
        dl_reliability = np.array([pred[0] >= reliability_th for pred in dl_predictions])
        if np.any(dl_reliability):
            crit_distance = horizontal_dist[np.max(dl_reliability.nonzero())]
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "DL", "Height": height, "Critical_Distance": crit_distance})
        else:
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "DL", "Height": height, "Critical_Distance": 0})
        # Evaluate UL reliability at each horizontal_dist
        ul_predictions = ul_model.predict(inputs)
        ul_reliability = np.array([pred[0] >= reliability_th for pred in ul_predictions])
        if np.any(ul_reliability):
            crit_distance = horizontal_dist[np.max(ul_reliability.nonzero())]
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "UL", "Height": height, "Critical_Distance": crit_distance})
        else:
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "UL", "Height": height, "Critical_Distance": 0})
        # Evaluate UL reliability at each horizontal_dist
        vid_predictions = vid_model.predict(inputs)
        vid_reliability = np.array([pred[0] >= reliability_th for pred in vid_predictions])
        if np.any(vid_reliability):
            crit_distance = horizontal_dist[np.max(vid_reliability.nonzero())]
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "VID", "Height": height, "Critical_Distance": crit_distance})
        else:
            crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "VID", "Height": height, "Critical_Distance": 0})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("Critical_Distances_DNN_Predictions_v2.csv", index=False)

In [53]:
pred = dl_model.predict(inputs)
pred = np.array([p[0] for p in pred])
# pred = np.append(pred, 0.99)
# pred = np.append(pred, 0.99)
if np.any(pred>=0.99):
    print(np.argmax(pred>=0.99))

4/4 [==============================] - 0s 2ms/step


## Find Critical Distance Using Simulated Reliability Version 2 </br>
Takes last occurence of h_dist that did not satisfy reliability threshold as crit_distance

In [16]:
dl_sim_df = pd.read_csv("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/data_processed/DJI_Spark_Downlink_Reliability.csv")
ul_sim_df = pd.read_csv("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/data_processed/DJI_Spark_Uplink_Reliability.csv")
vid_sim_df = pd.read_csv("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/data_processed/DJI_Spark_Video_Reliability.csv")

# Get the reliabilities for DL, UL and Video, and record it in dl_sim_df
dl_sim_df["Reliability_DL"] = dl_sim_df["Num_Reliable"] / dl_sim_df["Num_Sent"]
dl_sim_df["Reliability_UL"] = ul_sim_df["Num_Reliable"] / ul_sim_df["Num_Sent"]
dl_sim_df["Reliability_Vid"] = vid_sim_df["Num_Reliable"] / vid_sim_df["Num_Sent"]

dl_sim_df = get_mcs_index(dl_sim_df)

uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]
horizontal_dist = np.linspace(0, 1200, 121, endpoint=True)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
reliability_th = 0.99 # Threshold for reliability value
crit_dist_list = []
# for usi, mcs, height in tqdm(list(product(uav_send_int, mcs_index, heights))):
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    for height in heights:
        # Get the max hdist in dl_sim_df that fulfills reliability_th
        reliable_df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs) & (dl_sim_df["Reliability_DL"]>=reliability_th)]
        if not reliable_df.empty:
            crit_distance = reliable_df["Horizontal_Distance"].max()
        else:
            crit_distance = 0
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "DL", "Height": height, "Critical_Distance": crit_distance})

        # Get the max hdist in ul_sim_df that fulfills reliability_th
        reliable_df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs) & (dl_sim_df["Reliability_UL"]>=reliability_th)]
        if not reliable_df.empty:
            crit_distance = reliable_df["Horizontal_Distance"].max()
        else:
            crit_distance = 0
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "UL", "Height": height, "Critical_Distance": crit_distance})

        # Get the max hdist in ul_sim_df that fulfills reliability_th
        reliable_df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs) & (dl_sim_df["Reliability_Vid"]>=reliability_th)]
        if not reliable_df.empty:
            crit_distance = reliable_df["Horizontal_Distance"].max()
        else:
            crit_distance = 0
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Link": "VID", "Height": height, "Critical_Distance": crit_distance})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Critical_Distances_RelTh99_Simulated_Reliability_v2.csv", index=False)

  6%|▋         | 2/32 [00:00<00:01, 16.93it/s]

100%|██████████| 32/32 [00:01<00:00, 16.13it/s]


## Find D_max Using Simulated Reliability (For Training)
Takes first occurence of h_dist that did not satisfy reliability threshold FOR ALL LINKS as d_max

In [11]:
dl_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Downlink_Reliability.csv")
ul_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Uplink_Reliability.csv")
vid_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Video_Reliability.csv")

dl_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Downlink_Reliability.csv")
ul_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Uplink_Reliability.csv")
vid_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Video_Reliability.csv")

dl_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Downlink_Reliability.csv")
ul_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Uplink_Reliability.csv")
vid_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Video_Reliability.csv")

dl_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Downlink_Reliability.csv")
ul_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Uplink_Reliability.csv")
vid_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Video_Reliability.csv")
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]

# For the special case of QAM-16 26 Mbps, 10 ms USI, replace the values in "old" dataset with "newer" results
dl_sim_df_3 = dl_sim_df_3.loc[~((dl_sim_df_3["Bitrate"] == 26) & (dl_sim_df_3["UAV_Sending_Interval"] == 10) & (dl_sim_df_3["Height"].isin(heights)))]
ul_sim_df_3 = ul_sim_df_3.loc[~((ul_sim_df_3["Bitrate"] == 26) & (ul_sim_df_3["UAV_Sending_Interval"] == 10) & (ul_sim_df_3["Height"].isin(heights)))]
vid_sim_df_3 = vid_sim_df_3.loc[~((vid_sim_df_3["Bitrate"] == 26) & (vid_sim_df_3["UAV_Sending_Interval"] == 10) & (vid_sim_df_3["Height"].isin(heights)))]

# # For the special case of QPSK 13 Mbps, 20 ms USI, replace the values in "old" dataset with "newer" results
dl_sim_df_3 = dl_sim_df_3.loc[~((dl_sim_df_3["Bitrate"] == 13) & (dl_sim_df_3["UAV_Sending_Interval"] == 20) & (dl_sim_df_3["Height"].isin(heights)))]
ul_sim_df_3 = ul_sim_df_3.loc[~((ul_sim_df_3["Bitrate"] == 13) & (ul_sim_df_3["UAV_Sending_Interval"] == 20) & (ul_sim_df_3["Height"].isin(heights)))]
vid_sim_df_3 = vid_sim_df_3.loc[~((vid_sim_df_3["Bitrate"] == 13) & (vid_sim_df_3["UAV_Sending_Interval"] == 20) & (vid_sim_df_3["Height"].isin(heights)))]

dl = pd.concat((dl_sim_df_1, dl_sim_df_2, dl_sim_df_3, dl_sim_df_4))
ul = pd.concat((ul_sim_df_1, ul_sim_df_2, ul_sim_df_3, ul_sim_df_4))
vid = pd.concat((vid_sim_df_1, vid_sim_df_2, vid_sim_df_3, vid_sim_df_4))

dl.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)
ul.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)
vid.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)

dl.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Downlink_Reliability.csv")
ul.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Uplink_Reliability.csv")
vid.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Video_Reliability.csv")

In [ ]:
dl_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Downlink_Reliability.csv")
ul_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Uplink_Reliability.csv")
vid_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Video_Reliability.csv")

# Get the reliabilities for DL, UL and Video, and record it in dl_sim_df
dl_sim_df["Reliability_DL"] = dl_sim_df["Num_Reliable"] / dl_sim_df["Num_Sent"]
dl_sim_df["Reliability_UL"] = ul_sim_df["Num_Reliable"] / ul_sim_df["Num_Sent"]
dl_sim_df["Reliability_Vid"] = vid_sim_df["Num_Reliable"] / vid_sim_df["Num_Sent"]

dl_sim_df = get_mcs_index(dl_sim_df)

uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [60, 70, 90, 100, 120, 130, 150, 160, 180, 190, 210, 220, 240, 250, 270, 280, 300]
# horizontal_dist = np.linspace(0, 500, 51, endpoint=True) # Just use the hdist that's available (ASSUMING IT STARTS AT 0)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
reliability_th = 0.999 # Threshold for reliability value
crit_dist_list = []
# for usi, mcs, height in tqdm(list(product(uav_send_int, mcs_index, heights))):
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    for height in heights:
        crit_distance = 0
        df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs)]
        if (not df.empty):
            hdist_range = df["Horizontal_Distance"].max()
            horizontal_dist = np.arange(0, hdist_range + 10, step=10) # Assumes step size of 10
            for hdist in horizontal_dist:
                df_hdist = df.loc[df["Horizontal_Distance"]==hdist]
                if (df_hdist["Reliability_DL"].values[0] >= reliability_th) & (df_hdist["Reliability_UL"].values[0] >= reliability_th) & (df_hdist["Reliability_Vid"].values[0] >= reliability_th):
                    crit_distance = hdist
                else:
                    break
            if crit_distance == hdist_range:
                print("HDist limit reached. MCS_Index: {}, USI: {}, Height: {}, D_max: {}".format(mcs, usi, height, crit_distance))
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Height": height, "D_max": crit_distance})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/manual_control_max_hdist/Dmax_RelTh999_Simulated_Reliability_v3_10e5.csv", index=False)

  0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 32/32 [00:03<00:00,  9.55it/s]


## Find D_max Using Simulated Reliability (For Testing)
Takes first occurence of h_dist that did not satisfy reliability threshold FOR ALL LINKS as d_max

In [24]:
dl_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/mdp_max_hdist/data_processed/Downlink_Reliability.csv")
ul_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/mdp_max_hdist/data_processed/Uplink_Reliability.csv")
vid_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/mdp_max_hdist/data_processed/Video_Reliability.csv")

# Get the reliabilities for DL, UL and Video, and record it in dl_sim_df
dl_sim_df["Reliability_DL"] = dl_sim_df["Num_Reliable"] / dl_sim_df["Num_Sent"]
dl_sim_df["Reliability_UL"] = ul_sim_df["Num_Reliable"] / ul_sim_df["Num_Sent"]
dl_sim_df["Reliability_Vid"] = vid_sim_df["Num_Reliable"] / vid_sim_df["Num_Sent"]

dl_sim_df = get_mcs_index(dl_sim_df)

uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [75, 105, 135, 165, 195, 225, 255, 285]
horizontal_dist = np.linspace(0, 600, 61, endpoint=True)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
reliability_th = 0.99 # Threshold for reliability value
crit_dist_list = []

# for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
#     # First, check if the OCSVM model exist for this combination of usi and mcs
#     if os.path.exists(f"/media/research-student/KingstonSSD/ocsvm_models/ocsvm_Downlink_USI-{usi}_MCS-{mcs}.pkl"):
#         for height in heights:
#             crit_distance = 0
#             df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs)]
#             for hdist in horizontal_dist:
#                 df_hdist = df.loc[df["Horizontal_Distance"]==hdist]
#                 if (df_hdist["Reliability_DL"].values[0] >= reliability_th) & (df_hdist["Reliability_UL"].values[0] >= reliability_th) & (df_hdist["Reliability_Vid"].values[0] >= reliability_th):
#                     crit_distance = hdist
#                 else:
#                     break
#             if crit_distance > 0:
#                 crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Height": height, "D_max": crit_distance})

for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
   for height in heights:
        crit_distance = 0
        df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs)]
        if (not df.empty):
            hdist_range = df["Horizontal_Distance"].max()
            horizontal_dist = np.arange(0, hdist_range + 10, step=10) # Assumes step size of 10
            for hdist in horizontal_dist:
                df_hdist = df.loc[df["Horizontal_Distance"]==hdist]
                if (df_hdist["Reliability_DL"].values[0] >= reliability_th) & (df_hdist["Reliability_UL"].values[0] >= reliability_th) & (df_hdist["Reliability_Vid"].values[0] >= reliability_th):
                    crit_distance = hdist
                else:
                    break
            if crit_distance == hdist_range:
                print("HDist limit reached. MCS_Index: {}, USI: {}, Height: {}, D_max: {}".format(mcs, usi, height, crit_distance))
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Height": height, "D_max": crit_distance})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.sort_values(by=["USI", "MCS_Index", "Height"], inplace=True, ascending=[True, True, True])
crit_dist_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/manual_control_max_hdist/Testing_Dmax_RelTh99_Simulated_Reliability_v3_10e5.csv", index=False)

 16%|█▌        | 5/32 [00:00<00:00, 42.77it/s]

100%|██████████| 32/32 [00:01<00:00, 26.31it/s]


## (OLD) Filtering OCSVM Dataset using Critical Distance from CSV

In [5]:
DATASET_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/using_sim_reliability/data_ocsvm_train_processed" 
CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Dmax_RelTh99_Simulated_Reliability_v3_10e5.csv" # Set to "" to calc critical distances for each scenario

crit_dist_df = pd.read_csv(CRIT_DIST_FILE)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
throughput_df_details = [] 

dl_throughput_df = get_measured_throughput(DATASET_PATH, "Downlink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = dl_throughput_df.loc[(dl_throughput_df["UAV_Sending_Interval"]==usi) & (dl_throughput_df["MCS_Index"]==mcs)]
    context_crit_dist_df = crit_dist_df.loc[(crit_dist_df["USI"]==usi) & (crit_dist_df["MCS_Index"]==mcs) & (crit_dist_df["Link"]=="DL")]
    # For each height, get the data up to the critical distance
    df = []
    for h, cd in zip(context_crit_dist_df["Height"].values, context_crit_dist_df["Critical_Distance"].values):
        tmp_df = throughput_df.loc[(throughput_df["Height"]==h) & (throughput_df["Horizontal_Distance"]<=cd)]
        if not tmp_df.empty:
            df.append(tmp_df)
    if len(df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = pd.concat(df)
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "DL", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "DL", "Num_Samples": 0})

ul_throughput_df = get_measured_throughput(DATASET_PATH, "Uplink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = ul_throughput_df.loc[(ul_throughput_df["UAV_Sending_Interval"]==usi) & (ul_throughput_df["MCS_Index"]==mcs)]
    context_crit_dist_df = crit_dist_df.loc[(crit_dist_df["USI"]==usi) & (crit_dist_df["MCS_Index"]==mcs) & (crit_dist_df["Link"]=="UL")]
    # For each height, get the data up to the critical distance
    df = []
    for h, cd in zip(context_crit_dist_df["Height"].values, context_crit_dist_df["Critical_Distance"].values):
        tmp_df = throughput_df.loc[(throughput_df["Height"]==h) & (throughput_df["Horizontal_Distance"]<=cd)]
        if not tmp_df.empty:
            df.append(tmp_df)
    if len(df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = pd.concat(df)
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "UL", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "UL", "Num_Samples": 0})

vid_throughput_df = get_measured_throughput(DATASET_PATH, "Video")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = vid_throughput_df.loc[(vid_throughput_df["UAV_Sending_Interval"]==usi) & (vid_throughput_df["MCS_Index"]==mcs)]
    context_crit_dist_df = crit_dist_df.loc[(crit_dist_df["USI"]==usi) & (crit_dist_df["MCS_Index"]==mcs) & (crit_dist_df["Link"]=="VID")]
    # For each height, get the data up to the critical distance
    df = []
    for h, cd in zip(context_crit_dist_df["Height"].values, context_crit_dist_df["Critical_Distance"].values):
        tmp_df = throughput_df.loc[(throughput_df["Height"]==h) & (throughput_df["Horizontal_Distance"]<=cd)]
        if not tmp_df.empty:
            df.append(tmp_df)
    if len(df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = pd.concat(df)
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "VID", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "VID", "Num_Samples": 0})

throughput_df_details_df = pd.DataFrame(throughput_df_details)
throughput_df_details_df.to_csv("DJISpark_Measured_Throughput_100000Samples_Dataset_FilterBySim.csv")

  0%|          | 1/539 [00:00<02:39,  3.38it/s]

100%|██████████| 32/32 [00:00<00:00, 47.51it/s]


## Filtering By Measured Reliability

In [1]:
DL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_dl/model.010-0.2039.h5"
UL_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_ul/model.010-0.1202.h5"
VID_MODEL_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_vid/model.010-0.2581.h5"
DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/DJISpark_Measured_Throughput_10000Samples/data_processed" 
RELIABILITY_TH = 0.99

uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
throughput_df_details = [] 

dl_throughput_df = get_measured_throughput(DATASET_PATH, "Downlink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = dl_throughput_df.loc[(dl_throughput_df["UAV_Sending_Interval"]==usi) & (dl_throughput_df["MCS_Index"]==mcs)]
    if len(throughput_df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = throughput_df.loc[(throughput_df["Measured_Reliability"] >= RELIABILITY_TH)]
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "DL", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "DL", "Num_Samples": 0})

ul_throughput_df = get_measured_throughput(DATASET_PATH, "Uplink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = ul_throughput_df.loc[(ul_throughput_df["UAV_Sending_Interval"]==usi) & (ul_throughput_df["MCS_Index"]==mcs)]
    if len(throughput_df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = throughput_df.loc[(throughput_df["Measured_Reliability"] >= RELIABILITY_TH)]
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "UL", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "UL", "Num_Samples": 0})

vid_throughput_df = get_measured_throughput(DATASET_PATH, "Video")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = vid_throughput_df.loc[(vid_throughput_df["UAV_Sending_Interval"]==usi) & (vid_throughput_df["MCS_Index"]==mcs)]
    if len(throughput_df) > 0: # If data exist for this case, process it and append to throughput_df_list
        throughput_df = throughput_df.loc[(throughput_df["Measured_Reliability"] >= RELIABILITY_TH)]
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "VID", "Num_Samples": len(throughput_df)})
    else:
        throughput_df_details.append({"USI": usi, "MCS": mcs, "Link": "VID", "Num_Samples": 0})

throughput_df_details_df = pd.DataFrame(throughput_df_details)
throughput_df_details_df.to_csv("DJISpark_Measured_Throughput_10000Samples_Dataset_FilterByMeasuredReliability.csv")

100%|██████████| 32/32 [00:39<00:00,  1.23s/it]


## Find Critical Distance Using Measured Reliability

In [27]:
DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/DJISpark_Measured_Throughput_10000Samples/data_processed" 
RELIABILITY_TH = 0.99
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
crit_dist_list = [] 

dl_throughput_df = get_measured_throughput(DATASET_PATH, "Downlink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = dl_throughput_df.loc[(dl_throughput_df["UAV_Sending_Interval"]==usi) & (dl_throughput_df["MCS_Index"]==mcs)]
    # For each height, get the data up to the critical distance
    for height in heights:
        tmp_df = throughput_df.loc[(throughput_df["Height"]==height) & (throughput_df["Measured_Reliability"]>=RELIABILITY_TH)]
        if not tmp_df.empty:
            crit_dist = tmp_df["Horizontal_Distance"].max()
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "DL", "Height": height, "Critial_Distance": crit_dist})
        else:
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "DL", "Height": height, "Critial_Distance": 0})

ul_throughput_df = get_measured_throughput(DATASET_PATH, "Uplink")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = ul_throughput_df.loc[(ul_throughput_df["UAV_Sending_Interval"]==usi) & (ul_throughput_df["MCS_Index"]==mcs)]
    # For each height, get the data up to the critical distance
    for height in heights:
        tmp_df = throughput_df.loc[(throughput_df["Height"]==height) & (throughput_df["Measured_Reliability"]>=RELIABILITY_TH)]
        if not tmp_df.empty:
            crit_dist = tmp_df["Horizontal_Distance"].max()
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "UL", "Height": height, "Critial_Distance": crit_dist})
        else:
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "UL", "Height": height, "Critial_Distance": 0})

vid_throughput_df = get_measured_throughput(DATASET_PATH, "Video")
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    # Filter by USI and MCS
    throughput_df = vid_throughput_df.loc[(vid_throughput_df["UAV_Sending_Interval"]==usi) & (vid_throughput_df["MCS_Index"]==mcs)]
    # For each height, get the data up to the critical distance
    for height in heights:
        tmp_df = throughput_df.loc[(throughput_df["Height"]==height) & (throughput_df["Measured_Reliability"]>=RELIABILITY_TH)]
        if not tmp_df.empty:
            crit_dist = tmp_df["Horizontal_Distance"].max()
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "VID", "Height": height, "Critial_Distance": crit_dist})
        else:
            crit_dist_list.append({"USI": usi, "MCS_Index": mcs, "Link": "VID", "Height": height, "Critial_Distance": 0})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("Critical_Distance_by_Measured_Reliability.csv")

100%|██████████| 32/32 [00:40<00:00,  1.27s/it]
